<a href="https://colab.research.google.com/github/Rohanvernekarr/Gen-AI-Assignment-Task/blob/main/Task_3_Build_a_Chatbot_using_Hugging_Face_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A chatbot is a conversational AI system capable of interacting with users using natural language. In this assignment, we build a chatbot using a pre-trained transformer model from Hugging Face. Transformers are deep learning models that understand context and generate meaningful responses.

Import libraries

In [23]:
import os
# Disable ALL progress bars
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["DISABLE_TQDM"] = "1"
from transformers.utils import logging
logging.set_verbosity_error()

In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

In [25]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers.utils import logging
logging.set_verbosity_error()

In [26]:
!pip install transformers torch --quiet

In [27]:
# Standard library imports
import warnings
warnings.filterwarnings('ignore')
import torch
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully.
Device map: Single device


Response fn

In [28]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,
    )
    # Tokenize the prompt
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)
    # Generate the model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()
    # Append assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

In [29]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    # Initialize chat history with a system message to set behavior
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue
        # Generate reply from the model
        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        # fallback
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)
run_chatbot()

Chatbot: Hello! I am your AI assistant. How can I help you today?
User: Hello
Chatbot: Hello! How can I assist you today?
User: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) is a branch of computer science that aims to create intelligent machines capable of performing tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation. AI encompasses various techniques and algorithms, including machine learning, neural networks, natural language processing, and expert systems, which enable computers to learn from data, improve their performance on specific tasks, and make decisions without explicit programming. AI has wide-ranging applications in fields like healthcare, finance, transportation, and entertainment.
User: Who created Python?  
Chatbot: Python was created by Guido van Rossum in 1989 at the Centrum Wiskunde & Informatica (CWI) in the Netherlands. It was initially developed as a su